# Task 1 -- Image-to-FEN

Chessboard understanding with Qwen3.5-0.8B and token-level patch reordering (REOrder). Train on Colab; every training run and evaluation is cached on the Hugging Face Hub, so re-running a cell skips work already done.

Downloading Libraries and Imports

In [1]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv
!pip uninstall -y torchaudio torchao -q
!pip install -q flash-linear-attention



# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 55.5 MB/s eta 0:00:00


Setting up environment

In [ ]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        # /content survives "Restart session", so an old clone can linger -- pull so
        # the notebook always runs the latest code (restart the session afterwards if
        # a project module was already imported in this kernel).
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from eval.results import (
   evaluate_with_cache,
   load_or_init_results_table,
   append_or_replace,
   load_results_table,
   push_results_table,
)

from training.checkpoints import (
   find_resumable_checkpoint,
   train_or_load_finished,
   HubBestAdapterCallback,
)

from training.preprocessing import Qwen35OnTheFlyCollator

from training.reordering import (
   get_patch_reordering_indices,
   reorder_chessboard_image,
   apply_patch_permutation,
)

from training.finetune import finetune_and_push_chessboard_model

from training.learned_reordering import train_with_learned_reordering, load_learned_order, describe_order, greedy_order
from training.finetune import experiment_repo_id, experiment_model_name
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import install_token_reordering, verify_token_reordering
from eval.results import evaluate_with_order
from eval.diagnostics import (
    headline_table, fen_error_breakdown, analyze_san_predictions, san_error_breakdown,
    frame_ablation_losses, training_health_report, backfill_cached_metrics, save_diagnostics,
)
import gc

from eval.diagnostics import (
    backfill_cached_metrics,
    training_health_report,
    rescore_cached_fen_results,
    save_diagnostics,
)

# Used to key every cached evaluation result on the Hub (evaluation-results-task1
# and the per-strategy LoRA repos) -- defined here, early, instead of only inside
# the "Vanilla + LoRA" section further down, since evaluate_with_cache/
# load_or_init_results_table need it as soon as the very first evaluation runs.
hf_org_prefix = "bdatm-project"

print("All custom modules and eval utilities imported successfully!")

Downloading dataset for task 1 from HuggingFace repo

In [3]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name, num_proc=16)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

metadata.jsonl:   0%|          | 0.00/1.88M [00:00<?, ?B/s]

...

Setting num_proc from 16 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


Generating validation split:   0%|          | 0/400 [00:00<?, ? examples/s]

Setting num_proc from 16 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': 'drw7g', 'task': 'task1', 'fen': 'r3k2r/pppq2pp/2nb1p2/1B1Q3P/8/1P2P2P/PBPP1P2/R3K2R w KQkq - 1 13', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of a

## Experiment setup

**Token-level patch reordering (REOrder).** A scan order is applied to the visual *tokens* inside the model: each image's token embeddings and their M-RoPE positions are permuted together (`src/training/token_reordering.py`), so the board image is never altered and every token keeps its true 2D position. Only the order in which the LLM reads them changes -- which matters here because 18 of Qwen3.5's 24 LLM layers are Gated DeltaNet (a recurrent scan with no positional encoding).

The first runs of this notebook rearranged *pixel tiles* instead, which changed the board itself (a "file_wise" image was a transposed board). Their results stay in the results table, labelled `legacy: pixel-tile reordering, 512px`.

**Resolution (spec 3.3, patch alignment).** At `IMAGE_SIZE = 256` the LLM sees an 8x8 token grid: exactly one token per chess square. The legacy 512px raster run is the resolution ablation (512px = 2x2 tokens per square).

In [ ]:
IMAGE_SIZE = 256   # 256 -> 1 LLM token per square; 512 -> 2x2 tokens per square
TAG = f"tok{IMAGE_SIZE}"   # suffix of every Hub repo / results row of this setup
# Fixed scan orders (spec 3.3). "rank_wise" is the raster baseline itself;
# "hilbert" is one of REOrder's own orders (drop it to save a training run).
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3   # few-shot examples for the vanilla model, drawn from train
TASK = "task1"
model_id = "Qwen/Qwen3.5-0.8B"
hf_org_prefix = "bdatm-project"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto"
    )


def free_gpu():
    """Call after `del`-ing the model variables, so their GPU memory is released."""
    gc.collect()
    torch.cuda.empty_cache()


for s in ["raster", *STRATEGIES]:
    print(f"{s} (reading step of each square):\n{describe_order(resolve_square_order(s))}\n")

## Vanilla model

The processor is set to `IMAGE_SIZE` once here; training, evaluation and generation all go through it. `verify_token_reordering` checks on this environment's `transformers` version that no-order output equals the unpatched model, that embeddings and positions move together, and that the order actually changes the output.

In [ ]:
model_vanilla = load_base_model()
processor = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor, IMAGE_SIZE)
verify_token_reordering(model_vanilla, processor, dataset_task1["test"][0]["image"], resolve_square_order("zigzag"))

In [5]:
# Grab the first test sample
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Sample ID: sample_000000
FEN: 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33

Generating zero-shot prediction...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Predicted FEN (Zero-Shot): e2r1qppp1pppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppp


Zero-shot and few-shot evaluation of the vanilla model (spec, Member 1). Few-shot prepends `N_SHOT` solved train examples as earlier chat turns.

In [ ]:
all_models_results = load_or_init_results_table(task=TASK, hf_org_prefix=hf_org_prefix)

vanilla_results_df, vanilla_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name("Vanilla Qwen3.5-0.8B (Zero-Shot)", TAG),
)
all_models_results = append_or_replace(all_models_results, vanilla_summary_df)

few_shot_examples = [dataset_task1["train"][i] for i in range(N_SHOT)]
fewshot_results_df, fewshot_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name(f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", TAG),
    few_shot_examples=few_shot_examples,
)
all_models_results = append_or_replace(all_models_results, fewshot_summary_df)
display(fewshot_results_df.head() if fewshot_results_df is not None else fewshot_summary_df)
display(headline_table(all_models_results, TASK))
del model_vanilla
free_gpu()

## LoRA baseline (raster order)

Supervised LoRA fine-tuning with the native raster order (= rank-wise). Up to 10 epochs with early stopping on validation loss; the best epoch is kept, also across a Colab disconnect.

In [ ]:
raster_repo = experiment_repo_id(TASK, "raster", TAG, hf_org_prefix)
trained = finetune_and_push_chessboard_model(
    "raster", dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
del trained
free_gpu()

In [ ]:
raster_model = PeftModel.from_pretrained(load_base_model(), raster_repo)
raster_results_df, raster_summary_df = evaluate_with_order(
    TASK, raster_model, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix, adapter_repo_id=raster_repo,
    model_name=experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG),
)
all_models_results = append_or_replace(all_models_results, raster_summary_df)
display(headline_table(all_models_results, TASK))

## Training-free reordering

The raster-trained LoRA model reads the *same* images with a different token order -- no retraining. Since the board is intact, this measures how much the model depends on the order it was trained with.

In [ ]:
for strat in STRATEGIES:
    _, summary = evaluate_with_order(
        TASK, raster_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix,
        adapter_repo_id=raster_repo, model_name=experiment_model_name(f"Qwen + LoRA Raster, read {strat} (TF)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
display(headline_table(all_models_results, TASK))

## Fine-tuning with each scan order (SFT)

One LoRA run per strategy, trained and evaluated with that token order. Each run loads a fresh base model and frees it afterwards.

In [ ]:
for strat in STRATEGIES:
    trained = finetune_and_push_chessboard_model(
        strat, dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    )
    del trained
    free_gpu()

In [ ]:
for strat in STRATEGIES:
    repo = experiment_repo_id(TASK, strat, TAG, hf_org_prefix)
    strat_model = PeftModel.from_pretrained(load_base_model(), repo)
    _, summary = evaluate_with_order(
        TASK, strat_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix, adapter_repo_id=repo,
        model_name=experiment_model_name(f"Qwen + LoRA ({strat.capitalize()} - SFT)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
    del strat_model
    free_gpu()
display(headline_table(all_models_results, TASK))

## Learned reordering (REOrder)

REOrder's Plackett-Luce policy (`external/REOrder`) over the 64 squares, trained jointly with LoRA by REINFORCE (`src/training/learned_reordering.py`): one warm-up epoch in raster order, then the policy samples orders with a Gumbel temperature annealed 0.2 -> 0. Reward = minus the answer's cross-entropy; two orders are scored on each sample and compared with each other, so puzzle difficulty cancels out. The best epoch by validation loss is kept, together with the order it was validated with (the policy's greedy order, or raster if the warm-up epoch was best). Resumable from Hub checkpoints.

In [ ]:
learned_model, learned_policy, learned_repo, learned_order = train_with_learned_reordering(
    dataset_task1, processor, load_base_model(), peft_config, task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    num_train_epochs=4, policy_start_after_epochs=1,
)
print("Order used for evaluation (reading step of each square):")
print(describe_order(learned_order))
print("\nPolicy's greedy order:")
print(describe_order(greedy_order(learned_policy)))

_, learned_summary = evaluate_with_order(
    TASK, learned_model, processor, dataset_task1["test"], order=learned_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=learned_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering)", TAG),
)
all_models_results = append_or_replace(all_models_results, learned_summary)
display(headline_table(all_models_results, TASK))
del learned_model
free_gpu()

## Resolution ablation

Same raster LoRA recipe at 512px (legacy run: 2x2 tokens per square) vs 256px (1 token per square).

In [ ]:
LEGACY_RASTER_512 = "Qwen + LoRA Fine-Tuning (from HF)"   # raster LoRA of the first runs, 512px images
rows = all_models_results["model_name"].isin([LEGACY_RASTER_512, experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG)])
display(headline_table(all_models_results[rows], TASK))

## Results

Diagnostic columns are recomputed from the cached per-sample predictions (no inference), then the table is pushed to the Hub and reloaded from there. `headline_table` shows the spec's metrics first.

In [ ]:
all_models_results = backfill_cached_metrics(
    all_models_results, TASK, fens=None, hf_org_prefix=hf_org_prefix
)
results_repo = f"{hf_org_prefix}/evaluation-results-{TASK}"
push_results_table(results_repo, all_models_results, commit_message=f"{TASK}: comparison table")
display(headline_table(load_results_table(results_repo), TASK))

## Diagnostics

`training_health_report` per run: eval-loss history, discontinuities, recipe markers, best vs published epoch.

In [ ]:
if CONFIG["colab"]:
    from google.colab import drive
    drive.mount("/content/drive")   # save_diagnostics writes here, surviving a runtime reset

health = {r: training_health_report(r) for r in [experiment_repo_id(TASK, s, TAG, hf_org_prefix) for s in ["raster", *STRATEGIES, "learned"]]}
health_df = pd.DataFrame([
    {"repo_id": r, "n_problems": len(h["problems"]), "problems": " | ".join(h["problems"])} for r, h in health.items()
])
display(health_df)
save_diagnostics(health_df, f"training_health_{TAG}", TASK)

Qualitative error analysis of the raster LoRA model: how many squares are wrong per board, which pieces get confused, and examples.

In [ ]:
breakdown = fen_error_breakdown(raster_results_df)
display(breakdown["wrong_squares_per_board"])
display(breakdown["confusions"].head(15))
display(breakdown["examples"])
save_diagnostics(raster_results_df, f"raster_lora_per_sample_{TAG}", TASK)